# 02 — EDA & Visualization

**Objective:** Visualize the dataset to understand feature distributions and class imbalance.

- Class imbalance bar chart
- Feature correlation heatmap
- Amount & time distributions
- Fraud vs normal overlays

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from src.config import get_paths, get_model_config

paths = get_paths()
cfg = get_model_config()['preprocessing']
target_col = cfg['target_col']

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 12

In [ ]:
# Load raw data
df = pd.read_csv(paths['data']['raw_csv'])
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip("'\" ")
print(f'Shape: {df.shape}')

## 1. Class Imbalance Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute counts
class_counts = df[target_col].value_counts()
colors = ['#2ecc71', '#e74c3c']
class_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Class Distribution (Counts)', fontsize=14)
axes[0].set_xticklabels(['Normal (0)', 'Fraud (1)'], rotation=0)
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# Proportions
class_pct = df[target_col].value_counts(normalize=True) * 100
class_pct.plot(kind='bar', ax=axes[1], color=colors, edgecolor='black')
axes[1].set_title('Class Distribution (%)', fontsize=14)
axes[1].set_xticklabels(['Normal (0)', 'Fraud (1)'], rotation=0)
axes[1].set_ylabel('Percentage (%)')
for i, v in enumerate(class_pct):
    axes[1].text(i, v + 0.5, f'{v:.2f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(paths['reports']['figures_dir'], 'class_imbalance.png'), dpi=150)
plt.show()

## 2. Feature Correlation Heatmap

In [ ]:
# Select numeric columns for correlation
numeric_df = df.select_dtypes(include=[np.number])

fig, ax = plt.subplots(figsize=(12, 10))
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(paths['reports']['figures_dir'], 'correlation_heatmap.png'), dpi=150)
plt.show()

## 3. Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw amount distribution
df['amount'].hist(bins=100, ax=axes[0], color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Amount Distribution (Raw)', fontsize=14)
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Frequency')

# Log-transformed amount
np.log1p(df['amount']).hist(bins=100, ax=axes[1], color='coral', edgecolor='black', alpha=0.7)
axes[1].set_title('Amount Distribution (Log1p)', fontsize=14)
axes[1].set_xlabel('log(1 + Amount)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig(os.path.join(paths['reports']['figures_dir'], 'amount_distribution.png'), dpi=150)
plt.show()

## 4. Step (Time) Distribution

In [ ]:
step_col = cfg['step_col']

fig, ax = plt.subplots(figsize=(14, 5))
df.groupby(step_col)[target_col].agg(['count', 'sum']).plot(ax=ax)
ax.set_title('Transactions & Fraud over Time (Step)', fontsize=14)
ax.set_xlabel('Step')
ax.set_ylabel('Count')
ax.legend(['Total Transactions', 'Fraud Count'])
plt.tight_layout()
plt.savefig(os.path.join(paths['reports']['figures_dir'], 'step_distribution.png'), dpi=150)
plt.show()

## 5. Fraud vs Normal Overlays

In [ ]:
fraud = df[df[target_col] == 1]
normal = df[df[target_col] == 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Amount overlay
axes[0].hist(normal['amount'], bins=80, alpha=0.6, label='Normal', color='#2ecc71', density=True)
axes[0].hist(fraud['amount'], bins=80, alpha=0.7, label='Fraud', color='#e74c3c', density=True)
axes[0].set_title('Amount: Fraud vs Normal', fontsize=14)
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Density')
axes[0].legend()

# Step overlay
axes[1].hist(normal[step_col], bins=50, alpha=0.6, label='Normal', color='#2ecc71', density=True)
axes[1].hist(fraud[step_col], bins=50, alpha=0.7, label='Fraud', color='#e74c3c', density=True)
axes[1].set_title('Step (Time): Fraud vs Normal', fontsize=14)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(paths['reports']['figures_dir'], 'fraud_vs_normal_overlay.png'), dpi=150)
plt.show()

## 6. Category-wise Fraud Rate

In [ ]:
if 'category' in df.columns:
    cat_fraud = df.groupby('category')[target_col].agg(['mean', 'sum', 'count'])
    cat_fraud.columns = ['Fraud Rate', 'Fraud Count', 'Total']
    cat_fraud = cat_fraud.sort_values('Fraud Rate', ascending=False)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    cat_fraud['Fraud Rate'].plot(kind='bar', ax=ax, color='#e74c3c', edgecolor='black', alpha=0.8)
    ax.set_title('Fraud Rate by Category', fontsize=14)
    ax.set_ylabel('Fraud Rate')
    ax.set_xlabel('Category')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(os.path.join(paths['reports']['figures_dir'], 'fraud_rate_by_category.png'), dpi=150)
    plt.show()
    
    display(cat_fraud)